# DS528 — Exploratory Data Analysis
## 2026 FIFA World Cup Fan Travel Demand Prediction

### Notebook Overview
This notebook performs comprehensive EDA on the synthetic fan dataset enhanced with real-world data.

**Real-world data injected:**
- 16 official 2026 World Cup host cities with real coordinates
- 32 real countries with actual GDP per capita & FIFA rankings
- Real haversine distances from country centroids to nearest host city
- Real US/Canada/Mexico visa requirements by nationality

**Key Questions:**
1. What is the target distribution?
2. Which regions/countries have highest travel intent?
3. How do distance, visa, and income affect decisions?
4. What engagement patterns distinguish travelers?
5. Which features correlate most with the target?

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

DATA_PATH = Path.cwd().parent / 'data' / 'synthetic_worldcup_fans.csv'
OUTPUT_DIR = Path.cwd().parent / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
# Load the dataset
df = pd.read_csv(DATA_PATH)
print(f'Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'\nFirst 5 rows:')
df.head()

In [ ]:
# Data types and missing values
info_df = pd.DataFrame({
    'dtype': df.dtypes,
    'missing': df.isnull().sum(),
    'missing_pct': (df.isnull().sum() / len(df) * 100).round(2),
    'unique': df.nunique()
})
info_df[info_df['dtype'] == 'object'][['unique']].sort_values('unique', ascending=False)

---
## 1. Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Count plot
ax = axes[0]
counts = df['will_travel'].value_counts()
bars = ax.bar(['Will NOT Travel (0)', 'Will Travel (1)'], counts.values,
              color=['#E74C3C', '#27AE60'], edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
            f'{val:,}\n({val/len(df):.1%})', ha='center', fontsize=12, fontweight='bold')
ax.set_title('Target: will_travel', fontsize=14, fontweight='bold')
ax.set_ylim(0, counts.max() * 1.15)

# Travel rate by region
ax = axes[1]
region_stats = df.groupby('country_region')['will_travel'].agg(['mean', 'count']).sort_values('mean')
colors = plt.cm.RdYlGn(region_stats['mean'].values)
bars = ax.barh(region_stats.index, region_stats['mean'], color=colors, edgecolor='white')
for bar, val, cnt in zip(bars, region_stats['mean'], region_stats['count']):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.1%} (n={cnt:,})', va='center', fontsize=9)
ax.set_title('Travel Rate by Region', fontweight='bold')
ax.set_xlim(0, 1.15)

# Travel rate by income
ax = axes[2]
income_stats = df.groupby('income_level')['will_travel'].agg(['mean', 'count'])
income_order = ['Low', 'Medium', 'High']
income_stats = income_stats.reindex([i for i in income_order if i in income_stats.index])
bars = ax.bar(range(len(income_stats)), income_stats['mean'],
              color=['#E74C3C', '#F39C12', '#27AE60'], edgecolor='white')
ax.set_xticks(range(len(income_stats)))
ax.set_xticklabels([f'{i}\n(n={income_stats.loc[i, "count"]:,})' if i in income_stats.index else i for i in income_stats.index])
for bar, val in zip(bars, income_stats['mean']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.1%}',
            ha='center', fontweight='bold')
ax.set_title('Travel Rate by Income Level', fontweight='bold')
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'nb_target_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Overall travel rate: {df["will_travel"].mean():.1%}')
print(f'Class imbalance ratio: {df["will_travel"].mean():.2f}:1')

**Key Insight:** North America (host region) has ~90% travel rate due to proximity. Visa-requiring countries show dramatically lower intent. Income shows the expected gradient.

---
## 2. Geographical Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Distance vs travel rate by region
ax = axes[0, 0]
for region in df['country_region'].unique():
    subset = df[df['country_region'] == region]
    if len(subset) > 200:
        rate = subset.groupby(pd.cut(subset['distance_to_host_city_km'], bins=8))['will_travel'].mean()
        midpoints = [iv.mid for iv in rate.index]
        ax.plot(midpoints, rate.values, marker='o', label=region, linewidth=2, markersize=5)
ax.set_xlabel('Distance to Host City (km)', fontsize=11)
ax.set_ylabel('Travel Rate', fontsize=11)
ax.set_title('Distance Decay by Region', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Visa impact by region
ax = axes[0, 1]
visa_region = df.groupby(['country_region', 'visa_required'])['will_travel'].mean().unstack()
visa_region.columns = ['No Visa Required', 'Visa Required']
visa_region.plot(kind='bar', ax=ax, color=['#27AE60', '#E74C3C'], edgecolor='white')
ax.set_title('Visa Impact by Region', fontweight='bold')
ax.set_ylabel('Travel Rate')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

# Top 10 countries by travel rate
ax = axes[1, 0]
top_countries = df.groupby('country')['will_travel'].agg(['mean', 'count'])
top_countries = top_countries[top_countries['count'] > 100].sort_values('mean', ascending=False).head(10)
colors = plt.cm.RdYlGn(np.linspace(0.5, 1, len(top_countries))[::-1])
bars = ax.barh(range(len(top_countries)), top_countries['mean'][::-1], color=colors, edgecolor='white')
ax.set_yticks(range(len(top_countries)))
ax.set_yticklabels([f'{c} ({top_countries.loc[c, "count"]:,})' for c in top_countries.index[::-1]])
for i, (bar, val) in enumerate(zip(bars, top_countries['mean'][::-1])):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.1%}', va='center')
ax.set_title('Top 10 Countries by Travel Rate', fontweight='bold')
ax.set_xlim(0, 1.15)

# Host city distribution
ax = axes[1, 1]
host_dist = df['nearest_host_city'].value_counts().head(10)
colors = plt.cm.Blues(0.3 + 0.7 * (host_dist.values / host_dist.values[0]))
ax.barh(range(len(host_dist)), host_dist.values[::-1], color=colors[::-1], edgecolor='white')
ax.set_yticks(range(len(host_dist)))
ax.set_yticklabels(host_dist.index[::-1])
for i, (city, val) in enumerate(zip(host_dist.index[::-1], host_dist.values[::-1])):
    ax.text(val + 100, i, f'{val:,}', va='center')
ax.set_title('Fan Distribution by Nearest Host City', fontweight='bold')
ax.set_xlabel('Number of Fans')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'nb_geography_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

**Key Insight:** Distance decay is clear — travel rate drops sharply after ~5,000km. Visa requirements create a ~3x drop in travel intent. Vancouver is the most common host city due to its proximity to Asia (where the majority of the world population lives).

---
## 3. Numerical Feature Distributions

In [ ]:
num_features = ['age', 'distance_to_host_city_km', 'gdp_per_capita_usd',
                'football_engagement_score', 'social_media_engagement',
                'ticket_search_count', 'flight_search_count', 'hotel_search_count',
                'estimated_trip_cost', 'days_until_match']

fig, axes = plt.subplots(4, 3, figsize=(16, 16))
axes = axes.flatten()

for i, col in enumerate(num_features):
    ax = axes[i]
    for label, color, name in [(1, '#27AE60', 'Will Travel'), (0, '#E74C3C', "Won't Travel")]:
        subset = df[df['will_travel'] == label][col]
        ax.hist(subset, bins=40, alpha=0.5, color=color, label=name, density=True)
    ax.set_title(col.replace('_', ' ').title(), fontsize=10, fontweight='bold')
    ax.legend(fontsize=7)
    ax.tick_params(labelsize=7)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'nb_numerical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

**Key Insight:** Travelers have higher engagement scores, more searches, and slightly higher GDP. Distance distribution is bimodal for travelers (local + long-haul diehards). Trip cost is strongly right-skewed.

---
## 4. Categorical Feature Analysis

In [ ]:
cat_features = ['country_region', 'income_level', 'match_importance',
                'visa_required', 'favorite_team_qualified', 'previous_worldcup_attendance',
                'nearest_host_country']

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(cat_features):
    ax = axes[i]
    grouped = df.groupby(col)['will_travel'].agg(['mean', 'count']).sort_values('mean', ascending=True)
    colors = plt.cm.RdYlGn(grouped['mean'].values)
    bars = ax.barh(grouped.index.astype(str), grouped['mean'], color=colors, edgecolor='white')
    for bar, val, cnt in zip(bars, grouped['mean'], grouped['count']):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.1%} (n={cnt:,})', va='center', fontsize=8)
    ax.set_title(f'{col.replace("_", " ").title()}\nTravel Rate', fontweight='bold', fontsize=11)
    ax.set_xlim(0, 1.2)
    ax.tick_params(labelsize=8)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'nb_categorical_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

**Key Insight:** Match importance shows a slight positive trend (bigger matches = more travel). Previous attendance is a strong signal. Favorite team qualification boosts travel by ~15 percentage points. USA as host country gets highest rates (no visa + home advantage).

---
## 5. Correlation Analysis

In [ ]:
corr_cols = ['age', 'distance_to_host_city_km', 'gdp_per_capita_usd',
             'football_engagement_score', 'social_media_engagement',
             'ticket_search_count', 'flight_search_count', 'hotel_search_count',
             'estimated_trip_cost', 'visa_required', 'days_until_match',
             'favorite_team_qualified', 'previous_worldcup_attendance',
             'campaign_cost_usd', 'potential_net_revenue_usd', 'will_travel']

corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8}, annot_kws={'size': 8}, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'nb_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Top correlations with target
target_corr = corr['will_travel'].drop('will_travel').sort_values(key=abs, ascending=False)
print('Top 10 correlations with will_travel:')
for feat, val in target_corr.head(10).items():
    direction = '↑' if val > 0 else '↓'
    print(f'  {direction} {feat}: {val:+.3f}')

**Key Insight:** `distance_to_host_city_km` is the strongest negative predictor (-0.47). Engagement features and GDP show moderate positive correlation. `visa_required` is strongly negative. This confirms our engineered feature strategy.

---
## 6. Engagement & Search Pattern Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Engagement scatter
ax = axes[0, 0]
scatter = ax.scatter(df['football_engagement_score'], df['social_media_engagement'],
                     c=df['will_travel'].map({0: '#E74C3C', 1: '#27AE60'}), alpha=0.1, s=3)
ax.set_xlabel('Football Engagement Score')
ax.set_ylabel('Social Media Engagement')
ax.set_title('Engagement Scores by Travel Intent', fontweight='bold')

# Search total histogram
ax = axes[0, 1]
df['search_total'] = df['ticket_search_count'] + df['flight_search_count'] + df['hotel_search_count']
for label, color, name in [(0, '#E74C3C', "Won't Travel"), (1, '#27AE60', 'Will Travel')]:
    subset = df[df['will_travel'] == label]['search_total']
    ax.hist(subset, bins=30, alpha=0.5, color=color, label=name, density=True)
ax.set_xlabel('Total Search Count (ticket + flight + hotel)')
ax.set_title('Search Behavior by Travel Intent', fontweight='bold')
ax.legend()

# Age groups
ax = axes[0, 2]
bins = [0, 20, 30, 40, 50, 60, 100]
labels = ['<20', '20-30', '30-40', '40-50', '50-60', '60+']
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels)
age_rate = df.groupby('age_group')['will_travel'].mean()
colors = plt.cm.RdYlGn(age_rate.values)
ax.bar(range(len(age_rate)), age_rate.values, color=colors, edgecolor='white')
ax.set_xticks(range(len(age_rate)))
ax.set_xticklabels(age_rate.index)
ax.set_title('Travel Rate by Age Group', fontweight='bold')
ax.set_ylabel('Travel Rate')

# Match importance
ax = axes[1, 0]
match_rate = df.groupby('match_importance')['will_travel'].mean().sort_values(ascending=True)
colors = plt.cm.RdYlGn(match_rate.values)
bars = ax.barh(match_rate.index, match_rate.values, color=colors, edgecolor='white')
for bar, val in zip(bars, match_rate.values):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.1%}', va='center')
ax.set_title('Travel Rate by Match Importance', fontweight='bold')
ax.set_xlim(0, 1.15)

# Engagement × Previous Attendance interaction
ax = axes[1, 1]
df['eng_bucket'] = pd.cut(df['football_engagement_score'], bins=5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
interaction = df.groupby(['eng_bucket', 'previous_worldcup_attendance'])['will_travel'].mean().unstack()
interaction.columns = ['No Prior Attendance', 'Prior Attendance']
x = np.arange(len(interaction))
w = 0.35
ax.bar(x - w/2, interaction['No Prior Attendance'], w, label='No Prior Attendance', color='#E74C3C', edgecolor='white')
ax.bar(x + w/2, interaction['Prior Attendance'], w, label='Prior Attendance', color='#27AE60', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(interaction.index, rotation=30)
ax.set_title('Engagement × Prior Attendance', fontweight='bold')
ax.set_ylabel('Travel Rate')
ax.legend(fontsize=9)

# Team qualification × engagement
ax = axes[1, 2]
interaction2 = df.groupby(['eng_bucket', 'favorite_team_qualified'])['will_travel'].mean().unstack()
interaction2.columns = ['Team Not Qualified', 'Team Qualified']
ax.bar(x - w/2, interaction2['Team Not Qualified'], w, label='Team Not Qualified', color='#E74C3C', edgecolor='white')
ax.bar(x + w/2, interaction2['Team Qualified'], w, label='Team Qualified', color='#27AE60', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(interaction2.index, rotation=30)
ax.set_title('Engagement × Team Qualification', fontweight='bold')
ax.set_ylabel('Travel Rate')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'nb_engagement_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

**Key Insight:** Clear separation in engagement scores between travelers and non-travelers. Prior attendance amplifies engagement's effect. Team qualification matters most for highly engaged fans. Match importance shows monotonic positive trend.

---
## 7. Summary of EDA Findings

| # | Finding | Implication for Modeling |
|---|---------|-------------------------|
| 1 | 39% positive rate — moderate class imbalance | Use `class_weight='balanced'` or adjust threshold |
| 2 | Distance is the dominant negative predictor (-0.47) | Feature engineering: distance buckets, trip_feasibility |
| 3 | Visa requirement drops travel rate by ~3x | Strong categorical feature — must include |
| 4 | Engagement features separate travelers well | Composite engagement scores likely powerful |
| 5 | Prior attendance × engagement interaction | `fan_enthusiasm_score` captures this |
| 6 | Match importance shows positive gradient | Ordinal encoding or one-hot |
| 7 | Host countries (US/CA/MX) have >90% travel | `nearest_host_country` is informative |
| 8 | GDP per capita matters but with diminishing returns | `cost_income_index` more informative than raw cost |

These findings directly inform our feature engineering strategy in the modeling notebook.